# 01 — The Diagonal-Collapse Problem

Open quantum optimal transport with the question that makes it necessary: *if we reduce a quantum state to the probabilities a measurement returns, what do we throw away?*

**Prerequisites:** `03/14_benamou_brenier_otto` (Module 3 complete).

**What you'll be able to do**
- State the **diagonal-collapse problem**: distinct density matrices can share the same $Z$-basis diagonal.
- Build the canonical pair $|+\rangle\langle+|$ versus $I/2$ and confirm they share the diagonal $(\tfrac12, \tfrac12)$.
- Run classical optimal transport on those diagonals, watch it return $0$, and explain *why* it is blind here.

Welcome to Module 4 — quantum optimal transport. You arrive with the whole of classical OT behind you: Monge's map, the Kantorovich LP, the Wasserstein distance and its geodesics, Sinkhorn, the Bures bridge. This module lifts all of it to **density matrices**. Before we build the machinery, we earn the right to: this opening brick exposes, in one clean example, exactly what classical OT discards when it meets a quantum state. That gap is the reason the rest of the module exists.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum.density import (
    density_matrix,
    maximally_mixed,
    purity,
    von_neumann_entropy,
)
from qot_course.quantum.states import KET_PLUS
from qot_course.quantum_ot.preview import (
    diagonal_in_computational_basis,
    same_diagonal,
)
from qot_course.transport.discrete import (
    cityblock_cost,
    discrete_ot_plan,
    transport_cost,
)

np.random.seed(42)
viz.use_course_style()

## 1. A quantum state carries more than its measurement diagonal

Classical optimal transport takes two **probability vectors** $a, b \in \Delta^n$ and a ground cost, and returns a distance — that is the entire Module 3 story. A quantum state lives in a richer space: a density matrix $\rho$ on $\mathbb{C}^d$ is a $d \times d$ Hermitian, positive-semidefinite operator with unit trace. Its **diagonal** in the computational ($Z$) basis, $p_i = \rho_{ii}$, is a genuine probability vector — it is exactly what you get if you measure in that basis and tally the outcomes.

So a tempting shortcut suggests itself:

> **Naive reduction.** Read off the diagonal $p_i = \rho_{ii}$, treat it as a probability vector, and run classical OT. Surely that is enough?

It is not, and the reason is structural. The diagonal records only the *populations* — how often each basis outcome occurs. It says nothing about the **off-diagonal entries** $\rho_{ij}$ ($i \neq j$), the **coherences** that encode superposition and phase. Two states can agree on every population yet differ completely in their coherences. Classical OT, fed only the diagonal, **collapses** the state onto that one shadow and never sees the rest.

Let us make the gap concrete with the smallest possible example — a single qubit.

## 2. The canonical example — $|+\rangle\langle+|$ versus $I/2$

Take two qubit states that *feel* different and *are* different:

- $|+\rangle\langle+| = \begin{pmatrix} \tfrac12 & \tfrac12 \\ \tfrac12 & \tfrac12 \end{pmatrix}$ — the **pure** superposition $|+\rangle = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle)$, a single definite quantum state (von Neumann entropy $0$).
- $I/2 = \begin{pmatrix} \tfrac12 & 0 \\ 0 & \tfrac12 \end{pmatrix}$ — the **maximally mixed** state, an even classical coin flip between $|0\rangle$ and $|1\rangle$ (entropy $1$ bit).

One is the most coherent qubit state there is; the other carries no coherence at all. Yet look at the **diagonals**: both read $(\tfrac12, \tfrac12)$. Measured in the $Z$ basis, they are statistically indistinguishable — each gives $0$ and $1$ with probability one-half. The *only* difference lives off the diagonal: $|+\rangle\langle+|$ has coherences of $\tfrac12$ in the corners, $I/2$ has zeros.

We build both, extract their diagonals, confirm they match, and then hand those diagonals to classical OT.

In [ ]:
plus = density_matrix(KET_PLUS)   # |+><+| — pure superposition
mixed = maximally_mixed(2)        # I/2     — maximally mixed coin flip

diag_plus = diagonal_in_computational_basis(plus)
diag_mixed = diagonal_in_computational_basis(mixed)

print(f"Z-diagonal of |+><+| : {np.round(diag_plus, 12)}")
print(f"Z-diagonal of I/2    : {np.round(diag_mixed, 12)}")
print(f"same Z-diagonal?       {same_diagonal(plus, mixed)}  (tolerance 1e-9)")
print()

# These are genuinely different states — purity and entropy separate them cleanly.
print(f"purity tr(rho^2):  |+><+| = {purity(plus):.3f}   I/2 = {purity(mixed):.3f}")
print(f"von Neumann S:     |+><+| = {abs(von_neumann_entropy(plus)):.3f}   "
      f"I/2 = {von_neumann_entropy(mixed):.3f}  (bits)")

**Read the output.** The two diagonals are identical — `same_diagonal` returns `True` (using a 1e-9 tolerance that absorbs floating-point rounding) — so the naive reduction would hand classical OT *the very same probability vector* for both states. But the states are as far apart as a qubit allows: $|+\rangle\langle+|$ has purity $1$ and entropy $0$ (a single pure state), while $I/2$ has purity $\tfrac12$ and entropy $1$ bit (maximal disorder). The diagonal cannot tell them apart; purity and entropy have no trouble at all.

## 3. Classical OT on identical diagonals — the collapse in one number

Now the decisive step. Feed the two **identical** diagonals to classical optimal transport. We use the $1$-Wasserstein ground cost on the two outcomes $\{0, 1\}$ — `cityblock_cost` gives $C_{ij} = |i - j|$ — solve the Kantorovich LP with `discrete_ot_plan`, and read off the total transport cost.

In [ ]:
# Ground cost between the two measurement outcomes {0, 1}: C[i, j] = |i - j|.
cost = cityblock_cost([0, 1], [0, 1])
print("ground cost C:")
print(cost)
print()

# Classical OT on the two (identical) Z-diagonals.
plan = discrete_ot_plan(diag_plus, diag_mixed, cost)
w1_diagonals = transport_cost(plan, cost)
print(f"classical W_1 on the diagonals = {w1_diagonals:.4f}")
print()
print("Both states reduce to the same probability vector, so there is nothing for")
print("classical OT to move: it reports distance 0 and calls |+><+| and I/2 identical.")

**Read the output.** Classical OT returns **exactly $0$**. With identical source and target diagonals there is no mass to relocate, so the optimal plan moves nothing and the cost vanishes. By this measure $|+\rangle\langle+|$ and $I/2$ are *the same state* — a verdict we already know is wrong. This is the **diagonal-collapse problem** in a single number: collapse a quantum state onto its $Z$-diagonal and classical OT becomes blind to everything else.

To *see* what classical OT cannot, the two density matrices are plotted below — first $|+\rangle\langle+|$, then $I/2$.

In [ ]:
viz.plot_density_matrix(plus, title="$|+><+|$  —  pure superposition")
plt.show()

viz.plot_density_matrix(mixed, title="$I/2$  —  maximally mixed")
plt.show()

**Read the figures.** Each figure shows a density matrix as two annotated heatmaps — the real part on the left, the imaginary part on the right. Compare the **diagonals** (top-left and bottom-right cells) in the first figure and then in the second: both states read $0.5, 0.5$, the populations classical OT saw. Now compare the **off-diagonals** (top-right and bottom-left). For $|+\rangle\langle+|$ they are $0.5$ — the coherence of a genuine superposition. For $I/2$ they are $0$ — no coherence at all. The imaginary parts are zero for both here, so the entire difference between these states sits in those real off-diagonal corners.

That is precisely the structure the diagonal discards. The naive reduction keeps the two matching diagonals and erases the corners where the states disagree — which is why classical OT reported $0$. To respect this off-diagonal structure we need transport that acts on the *whole operator*, not on one shadow of it. Building that is the work of Module 4.

## Your turn

**Warm-up.** Build your own pair that shares a diagonal. Form the pure state $|\psi\rangle = \sqrt{0.7}\,|0\rangle + \sqrt{0.3}\,|1\rangle$ with `density_matrix`, and the classical mixture $\rho_{\text{mix}} = 0.7\,|0\rangle\langle0| + 0.3\,|1\rangle\langle1|$ with `mixed_state([KET_0, KET_1], [0.7, 0.3])`. Print both diagonals and check `same_diagonal`. *Before running the OT solver, predict what classical OT on the diagonals will return* — then confirm it. (For this you will import `mixed_state` from `qot_course.quantum.density`, and `KET_0`, `KET_1` from `qot_course.quantum.states`.)

**Go further.** Visualise that pair with `viz.plot_density_matrix`. Where do the two matrices agree, and where do they differ? Tie what you see back to the words *population* and *coherence* from §1.

**Challenge.** The diagonal-collapse problem hides the off-diagonal *coherences*. Construct a *different* failure of the diagonal: two states with the **same** diagonal $(\tfrac12, \tfrac12)$ that are **both pure** (so both have entropy $0$) yet are still different states. *Hint:* $|+\rangle\langle+|$ and $|{+}i\rangle\langle{+}i|$, with $|{+}i\rangle = \tfrac{1}{\sqrt2}(|0\rangle + i|1\rangle)$ — one stores its coherence in the real part, the other in the imaginary part. This is the doorway to the next notebook.

## What you built

- You stated the **diagonal-collapse problem**: a density matrix's $Z$-diagonal records only its populations, and distinct states can share it.
- You built the canonical pair $|+\rangle\langle+|$ and $I/2$ — opposite extremes of purity — and confirmed with `same_diagonal` that they carry the identical diagonal $(\tfrac12, \tfrac12)$.
- You ran classical OT on those diagonals, watched it return **$0$**, and saw in the density-matrix plots exactly where the discarded information lives: the off-diagonal coherences.

This is a real milestone. You have isolated, in the smallest possible system, exactly what classical optimal transport throws away when it meets a quantum state — the coherences that distinguish a pure superposition from a classical mixture. Naming the gap precisely is the first move toward closing it.

## What's next

In `02_noncommuting_same_diagonal` we sharpen the lesson. The $|+\rangle\langle+|$-vs-$I/2$ pair was, quietly, a *commuting* one. The deeper obstruction is that **non-commuting** states can share a diagonal too — and then there is no single basis in which both even *look* like probability vectors. That is the operator-level root of why a genuinely quantum transport is needed, and Module 4 sets out to build the machinery that can finally see the structure you exposed here.

## References

- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091
- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, Cambridge University Press (2010), ch. 2 — density matrices, purity, and the von Neumann entropy. DOI:10.1017/CBO9780511976667
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), chs. 2–3 — the discrete Kantorovich LP used here. DOI:10.1561/2200000073

**Previous:** `notebooks/03_ClassicalOptimalTransport/14_benamou_brenier_otto.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/02_noncommuting_same_diagonal.ipynb`